# Online-Payment Fraud Risk Decisioning

This PaySim capstone builds a Logistic Regression baseline that estimates fraud probability and supports approve, review, or decline decisions. The target is isFraud: 0 for legitimate and 1 for fraud.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)


from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import confusion_matrix, roc_curve, auc



RANDOM_STATE = 42
TEST_SIZE = 0.10             # Matches the referenced PaySim paper's 90:10 split
TUNING_FRACTION = 0.20       # Tune on 20% of training data, then refit on all training data
TARGET_RECALL = 0.80         # Example business constraint; validate with stakeholders

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')

## Data acquisition and quality

PaySim is a public synthetic transaction dataset. This section validates required fields, data types, missing values, duplicates, and severe target imbalance.

In [ ]:
df = pd.read_csv('data/paysim.csv')
df.columns = df.columns.map(str)
df["type"] = df["type"].astype(str)
df["isFraud"] = df["isFraud"].astype(int)

In [ ]:
required_columns = {
    'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest', 'isFraud'
}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f'Missing required columns: {sorted(missing_columns)}')

quality_summary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_count': df.isna().sum(),
    'missing_percent': 100 * df.isna().mean(),
    'unique_values': df.nunique(dropna=False),
})

print('Duplicate rows:', f'{df.duplicated().sum():,}')
print('Target values:', sorted(df['isFraud'].dropna().unique().tolist()))
display(quality_summary)

In [ ]:
class_summary = (
    df['isFraud']
    .value_counts(dropna=False)
    .rename_axis('isFraud')
    .to_frame('count')
)
class_summary['percent'] = 100 * class_summary['count'] / len(df)
display(class_summary)

fraud_rate = df['isFraud'].mean()
print(f'Fraud prevalence: {fraud_rate:.4%}')

## Exploratory analysis

Explore whether fraud is concentrated in specific transaction types and confirm the class imbalance. A log scale keeps rare fraud cases visible.

In [ ]:
plot_df = df[['type', 'isFraud']].copy()
plot_df['type'] = plot_df['type'].astype(str)
plot_df['isFraud'] = pd.to_numeric(plot_df['isFraud'], errors='raise').astype(int)

# Reindex guarantees that both target columns exist, even in a filtered sample.
transaction_counts = pd.crosstab(plot_df['type'], plot_df['isFraud']).reindex(
    columns=[0, 1], fill_value=0
)
transaction_counts.columns = ['legitimate', 'fraud']

fraud_by_type = pd.crosstab(
    plot_df['type'], plot_df['isFraud'], normalize='index'
).reindex(columns=[0, 1], fill_value=0)
fraud_by_type.columns = ['legitimate_rate', 'fraud_rate']
fraud_by_type = fraud_by_type.sort_values('fraud_rate', ascending=False)
display(fraud_by_type.round(6))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
transaction_counts.loc[fraud_by_type.index].plot(
    kind='bar', ax=axes[0], logy=True, color=['steelblue', 'firebrick']
)
axes[0].set_title('Transaction counts by type and fraud label (log scale)')
axes[0].set_xlabel('Transaction type')
axes[0].set_ylabel('Transaction count')
axes[0].tick_params(axis='x', rotation=30)

fraud_by_type['fraud_rate'].plot(kind='bar', ax=axes[1], color='firebrick')
axes[1].set_title('Observed fraud rate by transaction type')
axes[1].set_xlabel('Transaction type')
axes[1].set_ylabel('Fraud rate')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## Features and train-test split

Use transaction type, amount, and sender/receiver balances as predictors. Raw account identifiers are excluded initially, and stratification preserves fraud prevalence.

In [ ]:
categorical_features = ['type']
numeric_features = [
    'amount',
    'oldbalanceOrg',
    'newbalanceOrig',
    'oldbalanceDest',
    'newbalanceDest',
]
feature_columns = categorical_features + numeric_features

X = df[feature_columns].copy()
y = df['isFraud'].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print('Training shape:', X_train.shape)
print('Test shape:', X_test.shape)
print(f'Training fraud rate: {y_train.mean():.4%}')
print(f'Test fraud rate: {y_test.mean():.4%}')

## Preprocessing and model pipeline

The pipeline encodes and scales features, applies L1 feature selection, and fits Logistic Regression as an interpretable baseline for later model comparison.

In [ ]:
selector = make_column_selector(dtype_include=object)

extractor = SelectFromModel(
    LogisticRegression(
        penalty="l1",
        solver="liblinear",
        random_state=42
    )
)

transformer = make_column_transformer((OneHotEncoder(drop = 'first'), selector),
                                     remainder = StandardScaler())



lgr_pipe = Pipeline([('transformer', transformer),
                    ('selector', extractor),
                    ('lgr', LogisticRegression(random_state=42, max_iter = 1000))])

lgr_pipe.fit(X_train, y_train)

pipe_1_acc = lgr_pipe.score(X_test, y_test)

## Model evaluation

Evaluate precision, recall, F1, PR-AUC, ROC-AUC, false positives, and false negatives. Accuracy is secondary because fraud is rare.

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    RocCurveDisplay,
    auc as skl_auc
)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))

# Pipeline automatically performs transformation and feature selection
preds = lgr_pipe.predict(X_test)

conf_matrix = confusion_matrix(
    y_test,
    preds,
    labels=[0, 1]
)

ConfusionMatrixDisplay(
    confusion_matrix=conf_matrix,
    display_labels=["Legitimate", "Fraud"]
).plot(ax=ax[0], colorbar=False)

# Probability of the positive fraud class
fraud_scores = lgr_pipe.predict_proba(X_test)[:, 1]

fpr, tpr, thresholds = roc_curve(
    y_test,
    fraud_scores,
    pos_label=1
)

roc_auc_value = skl_auc(fpr, tpr)
pr_auc_value = average_precision_score(y_test, fraud_scores)

metric_summary = pd.Series({
    'accuracy': accuracy_score(y_test, preds),
    'precision': precision_score(y_test, preds, zero_division=0),
    'recall': recall_score(y_test, preds, zero_division=0),
    'f1': f1_score(y_test, preds, zero_division=0),
    'pr_auc': pr_auc_value,
    'roc_auc': roc_auc_value,
})

RocCurveDisplay(
    fpr=fpr,
    tpr=tpr,
    roc_auc=roc_auc_value
).plot(ax=ax[1])

fp = int(conf_matrix[0, 1])
fn = int(conf_matrix[1, 0])
auc = round(float(roc_auc_value), 2)

plt.tight_layout()
plt.show()

display(metric_summary.to_frame('value'))

fp, fn, auc

## Probability analysis

The 80% cutoff is an exploratory threshold. The final approve, review, or decline thresholds should balance fraud loss against false-positive customer friction.

In [ ]:
legitimate_probs = lgr_pipe.predict_proba(X_test)[:, 0]

high_prob_mask = legitimate_probs > 0.80
actual_legitimate = y_test.to_numpy() == 0

percent_of_test_data = high_prob_mask.mean()

percent_of_legitimate_captured = (
    (high_prob_mask & actual_legitimate).sum()
    / actual_legitimate.sum()
)

print("High-confidence legitimate transactions as % of test data:")
print(percent_of_test_data)

print("Actual legitimate transactions captured:")
print(percent_of_legitimate_captured)

In [ ]:
fraud_probs = lgr_pipe.predict_proba(X_test)[:, 1]
high_risk_mask = fraud_probs > 0.80
actual_fraud = y_test.to_numpy() == 1

percent_flagged = high_risk_mask.mean()

fraud_precision = (
    actual_fraud[high_risk_mask].mean()
    if high_risk_mask.any()
    else 0
)

fraud_recall = (
    (high_risk_mask & actual_fraud).sum()
    / actual_fraud.sum()
)

print("Percent of transactions flagged:", percent_flagged)
print("Fraud precision among flagged:", fraud_precision)
print("Percentage of actual fraud captured:", fraud_recall)

## Important features

Rank selected features by coefficient magnitude to explain the model. Coefficients show associations with predicted fraud, not causation.

In [ ]:
### GRADED

coef_df = ""

### BEGIN SOLUTION

# All feature names created by one-hot encoding and scaling
feature_names = np.asarray(
    lgr_pipe.named_steps["transformer"].get_feature_names_out(),
    dtype=str
)

# Boolean mask identifying features retained by SelectFromModel
selected_mask = lgr_pipe.named_steps["selector"].get_support()

selected_features = feature_names[selected_mask]

# Remove transformer prefixes such as onehotencoder__ and standardscaler__
clean_names = [
    feature.split("__", 1)[-1]
    for feature in selected_features
]

# Coefficients from the final Logistic Regression model
coefficients = lgr_pipe.named_steps["lgr"].coef_[0]

coef_df = pd.DataFrame({
    "feature": clean_names,
    "coefficient": coefficients,
    "coefs": np.abs(coefficients)
})

coef_df = coef_df.sort_values(
    by="coefs",
    ascending=False
).reset_index(drop=True)


## Capstone conclusion

Logistic Regression provides an interpretable baseline, not a production fraud claim. PaySim is synthetic; next steps are cross-validation, GridSearchCV, threshold optimization, and comparison with tree-based models.